In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LBCO+Si, McStas

This example demonstrates a Rietveld refinement of La0.5Ba0.5CoO3
crystal structure with a small amount of Si phase using time-of-flight
neutron powder diffraction data simulated with McStas.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structures

This section shows how to add structures and modify their
parameters.

### Create Structure 1: LBCO

In [3]:
structure_1 = StructureFactory.from_scratch(name='lbco')

#### Set Space Group

In [4]:
structure_1.space_group.name_h_m = 'P m -3 m'
structure_1.space_group.coord_system_code = '1'

#### Set Unit Cell

In [5]:
structure_1.cell.length_a = 3.8909

#### Set Atom Sites

In [6]:
structure_1.atom_sites.create(
    id='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.2,
    occupancy=0.5,
)
structure_1.atom_sites.create(
    id='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.2,
    occupancy=0.5,
)
structure_1.atom_sites.create(
    id='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=0.2567,
)
structure_1.atom_sites.create(
    id='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    adp_iso=1.4041,
)

### Create Structure 2: Si

In [7]:
structure_2 = StructureFactory.from_scratch(name='si')

#### Set Space Group

In [8]:
structure_2.space_group.name_h_m = 'F d -3 m'
structure_2.space_group.coord_system_code = '2'

#### Set Unit Cell

In [9]:
structure_2.cell.length_a = 5.43146

#### Set Atom Sites

In [10]:
structure_2.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.0,
    adp_iso=0.0,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Download Data

In [11]:
data_path = download_data('meas-lbco-si-mcstas', destination='data')

Getting data...


Data 'meas-lbco-si-mcstas': La0.5Ba0.5CoO3 + Si, McStas simulation


✅ Data 'meas-lbco-si-mcstas' downloaded to '../../../data/meas-lbco-si-mcstas.xye'


### Create Experiment

In [12]:
experiment = ExperimentFactory.from_data_path(
    name='mcstas',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='bragg',
)

### Set Instrument

In [13]:
experiment.instrument.setup_twotheta_bank = 94.90931761529106
experiment.instrument.calib_d_to_tof_linear = 58724.76869981215

### Set Peak Profile

In [14]:
experiment.peak.broad_gauss_sigma_0 = 45137
experiment.peak.broad_gauss_sigma_1 = -52394
experiment.peak.broad_gauss_sigma_2 = 22998
experiment.peak.decay_beta_0 = 0.0055
experiment.peak.decay_beta_1 = 0.0041
experiment.peak.rise_alpha_0 = 0
experiment.peak.rise_alpha_1 = 0.0097

### Set Background

Select the background type.

In [15]:
experiment.background.type = 'line-segment'

Background type for experiment 'mcstas' already set to


line-segment


Add background points.

In [16]:
experiment.background.create(id='1', position=45000, intensity=0.2)
experiment.background.create(id='2', position=50000, intensity=0.2)
experiment.background.create(id='3', position=55000, intensity=0.2)
experiment.background.create(id='4', position=65000, intensity=0.2)
experiment.background.create(id='5', position=70000, intensity=0.2)
experiment.background.create(id='6', position=75000, intensity=0.2)
experiment.background.create(id='7', position=80000, intensity=0.2)
experiment.background.create(id='8', position=85000, intensity=0.2)
experiment.background.create(id='9', position=90000, intensity=0.2)
experiment.background.create(id='10', position=95000, intensity=0.2)
experiment.background.create(id='11', position=100000, intensity=0.2)
experiment.background.create(id='12', position=105000, intensity=0.2)
experiment.background.create(id='13', position=110000, intensity=0.2)

### Set Linked Structures

In [17]:
experiment.linked_structures.create(structure_id='lbco', scale=4.0)
experiment.linked_structures.create(structure_id='si', scale=0.2)

## 📦 Define Project

The project object is used to manage structures, experiments, and
analysis.

### Create Project

In [18]:
project = Project(name='lbco_si_mcstas')

### Add Structures

In [19]:
project.structures.add(structure_1)
project.structures.add(structure_2)

### Show Structures

In [20]:
project.structures.show_names()

Defined structures 🧩


['lbco', 'si']


### Add Experiments

In [21]:
project.experiments.add(experiment)

### Display Structure

In [22]:
project.display.structure(struct_name='lbco')
project.display.structure(struct_name='si')

Structure 🧩 'lbco' (Atom view type: 'covalent')


Structure 🧩 'si' (Atom view type: 'covalent')


### Set Excluded Regions

Show measured data as loaded from the file.

In [23]:
project.display.pattern(expt_name='mcstas')

Add excluded regions.

In [24]:
experiment.excluded_regions.create(id='1', start=0, end=40000)
experiment.excluded_regions.create(id='2', start=108000, end=200000)

Show excluded regions.

In [25]:
experiment.excluded_regions.show()

Excluded regions


,start,end
1,0.00000,40000.00000
2,108000.00000,200000.00000


Show measured data after adding excluded regions.

In [26]:
project.display.pattern(expt_name='mcstas')

Show experiment as text.

In [27]:
project.experiments['mcstas'].show_as_text()

Experiment 🔬 'mcstas' as text


,Edi
1,data_mcstas
2,
3,_experiment_type.sample_form powder
4,_experiment_type.beam_mode time-of-flight
5,_experiment_type.radiation_probe neutron
6,_experiment_type.scattering_type bragg
7,
8,_diffrn.ambient_temperature ?
9,_diffrn.ambient_pressure ?
10,_diffrn.ambient_magnetic_field ?


## 🚀 Perform Analysis

This section outlines the analysis process, including how to configure
calculation and fitting engines.

### Set Free Parameters

Set structure parameters to be optimized.

In [28]:
structure_1.cell.length_a.free = True
structure_1.atom_sites['Co'].adp_iso.free = True
structure_1.atom_sites['O'].adp_iso.free = True

structure_2.cell.length_a.free = True

Set experiment parameters to be optimized.

In [29]:
experiment.linked_structures['lbco'].scale.free = True
experiment.linked_structures['si'].scale.free = True

experiment.peak.broad_gauss_sigma_0.free = True
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_gauss_sigma_2.free = True

experiment.peak.rise_alpha_1.free = True
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

for point in experiment.background:
    point.intensity.free = True

### Run Fitting

In [30]:
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

In [31]:
project.analysis.fit()
project.display.fit.results()
project.display.fit.correlations()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'mcstas' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.
4,gradient_tolerance,0.0,Gradient orthogonality used to stop fitting; zero disables it.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),9.25
4,🔁 Iterations,238
5,📏 Goodness-of-fit (reduced χ²),9.53
6,"📏 R-factor (Rf, %)",5.79
7,"📏 R-factor squared (Rf², %)",5.34
8,"📏 Weighted R-factor (wR, %)",7.63


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8909,3.8905,0.0002,0.01 % ↓
2,lbco,atom_site,Co,adp_iso,Å²,0.2567,0.2330,0.1430,9.24 % ↓
3,lbco,atom_site,O,adp_iso,Å²,1.4041,2.1176,0.0476,50.81 % ↑
4,si,cell,,length_a,Å,5.4315,5.4360,0.0019,0.08 % ↑
5,mcstas,linked_structure,lbco,scale,,4.0000,39.1433,0.4646,878.58 % ↑
6,mcstas,linked_structure,si,scale,,0.2000,0.0411,0.0035,79.43 % ↓
7,mcstas,peak,,rise_alpha_1,μs/Å,0.0097,0.0097,0.0004,0.50 % ↑
8,mcstas,peak,,decay_beta_0,μs,0.0055,0.0055,0.0002,0.73 % ↑
9,mcstas,peak,,decay_beta_1,μs/Å,0.0041,0.0043,0.0005,4.77 % ↑
10,mcstas,peak,,broad_gauss_sigma_0,μs²,45137.0000,39050.7814,4554.6213,13.48 % ↓


### Display Pattern

In [32]:
project.display.pattern(expt_name='mcstas')

## 💾 Save Project

In [33]:
project.save_as(dir_path='projects/refine-lbco-si-mcstas')

Saving project 📦 'lbco_si_mcstas' to '../../../projects/refine-lbco-si-mcstas'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 lbco.edi
│   └── 📄 si.edi
├── 📁 experiments/
│   └── 📄 mcstas.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 lbco_si_mcstas.html
